In [ ]:
from pathlib import Path
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from PIL import Image

# Device: Apple Silicon GPU (mps) or CPU
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Using compute device: {device}")


In [ ]:
# Dataset paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset" / "cats_and_dogs"

# 1. Resize every photo to 128x128 and convert to PyTorch numbers
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 2. Load images directly from folders
train_dataset = datasets.ImageFolder(str(DATASET_DIR / "train"), transform=transform)
val_dataset = datasets.ImageFolder(str(DATASET_DIR / "validation"), transform=transform)

SUBSET_SIZE = 2000
train_sub = Subset(train_dataset, random.sample(range(len(train_dataset)), SUBSET_SIZE))
val_sub = Subset(val_dataset, random.sample(range(len(val_dataset)), SUBSET_SIZE // 4))

# 4. Group images into batches of 32
train_loader = DataLoader(train_sub, batch_size=32, shuffle=True)
val_loader = DataLoader(val_sub, batch_size=32, shuffle=False)

class_names = train_dataset.classes 
print("Classes:", class_names)
print(f"Ready: {len(train_sub)} train images ({len(train_loader)} batches) | {len(val_sub)} val images ({len(val_loader)} batches)")


In [ ]:
# Simple CNN Architecture
class SimpleCatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        # PART 1: Find Clues (2 simple scanning rounds)
        self.features = nn.Sequential(
            # Round 1: 16 filters | shrinks 128x128 -> 64x64
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            # Round 2: 32 filters | shrinks 64x64 -> 32x32
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # PART 2: Make the Final Guess
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 2)  # Vote: Cat (0) or Dog (1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Put the model on Apple Silicon GPU (mps)
model = SimpleCatDogCNN().to(device)
print("Model ready on:", device)
print(model)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        
    print(f"Epoch {epoch + 1:2d}/20 | Loss: {loss.item():.4f}")

print("Training finished!")


In [ ]:
# Save the trained model locally
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "cat_dog_model.pth"

torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved locally at: {MODEL_PATH}")
